In [ ]:
pip install indic-transliteration

In [ ]:
import re

class TamilVenbaParser:
    def __init__(self):

        self.uyir_kuril = ['அ', 'இ', 'உ', 'எ', 'ஒ']
        self.uyir_nedil = ['ஆ', 'ஈ', 'ஊ', 'ஏ', 'ஐ', 'ஓ', 'ஔ']


        self.uyirmei_kuril_mod = ['ி', 'ு', 'ெ', 'ொ']
        self.uyirmei_nedil_mod = ['ா', 'ீ', 'ூ', 'ே', 'ை', 'ோ', 'ௌ']
        self.pulli = ['்']
        self.ayudham = ['ஃ']


        self.consonants = [
            'க', 'ங', 'ச', 'ஞ', 'ட', 'ண', 'த', 'ந', 'ப', 'ம',
            'ய', 'ர', 'ல', 'வ', 'ழ', 'ள', 'ற', 'ன',
            'ஶ', 'ஜ', 'ஷ', 'ஸ', 'ஹ'
        ]

        self.kutrialugaram = ['கு', 'சு', 'டு', 'து', 'பு', 'று']

    def map_kuril_nedil(self, tamil_word):
        """State machine to parse Tamil Unicode into 0 (Kuril), 1 (Nedil), 2 (Otru)."""
        mapped = ""
        chars = list(tamil_word)
        i = 0

        while i < len(chars):
            c = chars[i]
            if c in self.ayudham:
                mapped += '2'
                i += 1
            elif c in self.uyir_kuril:
                mapped += '0'
                i += 1
            elif c in self.uyir_nedil:
                mapped += '1'
                i += 1
            elif c in self.consonants:

                if i + 1 < len(chars):
                    next_char = chars[i+1]
                    if next_char in self.pulli:
                        mapped += '2'
                        i += 2
                        continue
                    elif next_char in self.uyirmei_nedil_mod:
                        mapped += '1'
                        i += 2
                        continue
                    elif next_char in self.uyirmei_kuril_mod:
                        mapped += '0'
                        i += 2
                        continue


                mapped += '0'
                i += 1
            else:
                i += 1


        return re.sub(r'2+', '2', mapped)

    def extract_asai(self, mapped_string):
        """Greedy parser to extract valid Ner/Nirai patterns from the mapped string."""
        asai_list = []
        i = 0
        n = len(mapped_string)

        while i < n:

            if i + 3 <= n and mapped_string[i:i+3] in ['002', '012']:
                asai_list.append("Nirai")
                i += 3
            elif i + 2 <= n and mapped_string[i:i+2] in ['00', '01']:
                asai_list.append("Nirai")
                i += 2

            elif i + 2 <= n and mapped_string[i:i+2] in ['02', '12']:
                asai_list.append("Ner")
                i += 2
            elif i + 1 <= n and mapped_string[i:i+1] in ['0', '1']:
                asai_list.append("Ner")
                i += 1
            else:
                i += 1

        return asai_list

    def determine_seer_type(self, asai_list, is_eetru_cheer=False, word=""):
        """Classifies the Vaaipadu based on the Asai sequence."""
        if is_eetru_cheer:
            if len(asai_list) == 1:
                return "Naal" if asai_list[0] == "Ner" else "Malar", "Eetru"
            elif len(asai_list) == 2:

                if any(word.endswith(k) for k in self.kutrialugaram):
                    return ("Kaasu", "Eetru") if asai_list == ["Ner", "Ner"] else ("Pirappu", "Eetru")
            return "Invalid Eetru", "Invalid"

        pattern = "-".join(asai_list)
        mapping = {
            "Ner-Ner": ("Thema", "Maa"),
            "Nirai-Ner": ("Pulima", "Maa"),
            "Ner-Nirai": ("Koovilam", "Vilam"),
            "Nirai-Nirai": ("Karuvilam", "Vilam"),
            "Ner-Ner-Ner": ("Themangai", "Kaai"),
            "Nirai-Ner-Ner": ("Pulimangai", "Kaai"),
            "Ner-Nirai-Ner": ("Koovilangai", "Kaai"),
            "Nirai-Nirai-Ner": ("Karuvilangai", "Kaai")
        }
        return mapping.get(pattern, ("Invalid Seer", "Invalid"))

    def check_thalai(self, prev_group, next_first_asai):
        """Validates the strict Thalai rules across adjacent seers."""
        if prev_group == "Maa" and next_first_asai == "Nirai":
            return True, "Iyarcheer Vendalai"
        if prev_group == "Vilam" and next_first_asai == "Ner":
            return True, "Iyarcheer Vendalai"
        if prev_group == "Kaai" and next_first_asai == "Ner":
            return True, "Venseer Vendalai"

        return False, f"Thalai Violation: '{prev_group}' ending before '{next_first_asai}'"

    def parse_venba(self, venba_text):
        """Main pipeline to validate structure, Alagiduthal, and Thalai."""


        cleaned_text = re.sub(r'[\u200B-\u200D\uFEFF]', '', venba_text)

        cleaned_text = re.sub(r'[^\u0B80-\u0BFF\s]', ' ', cleaned_text)

        lines = [line.strip() for line in cleaned_text.strip().split('\n') if line.strip()]
        line_count = len(lines)

        if not (2 <= line_count <= 12):
            return {"status": "Error", "message": f"Expected 2-12 lines, found {line_count}"}

        all_seers = []
        for i, line in enumerate(lines):
            words = line.split()
            expected = 3 if i == line_count - 1 else 4
            if len(words) != expected:
                return {"status": "Error", "message": f"Line {i+1} has {len(words)} seers. Expected {expected}."}
            all_seers.extend(words)

        parsed_data = []


        for i, word in enumerate(all_seers):
            is_eetru = (i == len(all_seers) - 1)
            mapped_str = self.map_kuril_nedil(word)
            asai_list = self.extract_asai(mapped_str)

            seer_name, seer_group = self.determine_seer_type(asai_list, is_eetru, word)

            if seer_name == "Invalid Seer":
                return {"status": "Error", "message": f"Grammatically invalid Asai sequence at '{word}'"}

            parsed_data.append({
                "word": word,
                "asai": asai_list,
                "name": seer_name,
                "group": seer_group
            })

        for i in range(len(parsed_data) - 1):
            prev_seer = parsed_data[i]
            next_seer = parsed_data[i+1]
            next_first_asai = next_seer["asai"][0]

            is_valid, rule = self.check_thalai(prev_seer["group"], next_first_asai)
            if not is_valid:
                msg = f"{rule} between '{prev_seer['word']}' ({prev_seer['name']}) and '{next_seer['word']}'."
                return {"status": "Error", "message": msg}

        return {"status": "Success", "data": parsed_data}



if __name__ == "__main__":
    parser = TamilVenbaParser()

    nalavenba = """அரிதவித்து ஆசின் றுணர்ந்தவன் பாதம்
விரிகடல் சூழ்ந்த வியன்கண்மா ஞாலத்து
உரியதனிற் கண்டுணர்ந்தார் ஓக்கமே போலப்
'பெரியதன் ஆவி பெரிது."""

    result = parser.parse_venba(nalavenba)

    if result["status"] == "Success":
        print("\n Valid Venba Structure. Alagiduthal Breakdown:\n")
        for item in result["data"]:
            print(f"{item['word']:<15} | {' / '.join(item['asai']):<25} | {item['name']}")
    else:
        print(f"\n Failed: {result['message']}")


 Failed: Thalai Violation: 'Kaai' ending before 'Nirai' between 'ஞாலத்து' (Themangai) and 'உரியதனிற்'.
